# Eerste trainingsproef: een MX-gewichtsmodus

CNN-8 / SVHN train-only. A32, FP32-updatemaster en Adam; alleen E2M3-gewichten in de zes doellagen.

Dit notebook leest uitsluitend opgeslagen rapporten. Geen training, modelcalls, downloads of wijzigingen aan ruwe onderzoeksgegevens. Vijf epochs zijn een technische/leertest, geen eindvergelijking met de paper. `control` is een **afzonderlijk** W32/A32-model, niet een extra precisie in dezelfde training.

In [ ]:
from pathlib import Path
RUN_DIR = Path.cwd()  # In de runkopie wordt dit automatisch ingevuld.
CONTROL_DIR = None  # Optioneel: Path(r'C:\...\control_run')

In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

required = ['run_report.json', 'epoch_metrics.csv', 'selected_metrics.csv', 'history.json']
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Geen volledige trainingsrapportage in {RUN_DIR}: {missing}")
def read_json(name, root=RUN_DIR):
    return json.loads((root / name).read_text(encoding='utf-8'))
report = read_json('run_report.json')
history = read_json('history.json')
config = report['config']
metrics = pd.read_csv(RUN_DIR / 'epoch_metrics.csv')
selected = pd.read_csv(RUN_DIR / 'selected_metrics.csv')
times = pd.read_csv(RUN_DIR / 'epoch_times.csv')
updates = pd.read_csv(RUN_DIR / 'weight_updates.csv')
synthetic = bool(report.get('synthetic_fixture_only', False))
if synthetic:
    display(Markdown('**LET OP: synthetische softwaretest; geen SVHN-resultaten.**'))
display(pd.DataFrame([{'status': report['status'], 'mode': config['mode'],
    'gewicht': config['weight_format'], 'activaties': config['activation_bits'],
    'master': config['master_dtype'], 'epochs': report['completed_epoch'],
    'updates': report['optimizer_steps'], 'distillatie': config['distillation'],
    'truncatie': config['truncation']}]))
display(selected[['role','epoch','weight_format','n','correct1','count_top1_percent','objective_loss']])

## Rapportcontroles en waarschuwingen
De beste epoch wordt geselecteerd met de actieve modus op de officiele testset; de testscore is daardoor niet onafhankelijk van selectie. BN32 leert in training, maar verandert niet tijdens evaluatie. Ongebruikte BN-banken blijven aanwezig en tellen mee in de checkpointopslag.

In [ ]:
records = history['epochs']
checks = {
    'Een train- en een testrij per epoch': len(metrics) == 2 * len(records),
    'Geen dubbele metriccombinaties': not metrics.duplicated(['epoch','phase']).any(),
    'Eindige loss en accuracies': all(math.isfinite(float(v)) for col in ['objective_loss','top1_percent'] for v in metrics[col]),
    'Correcte aantallen consistent': ((100*metrics.correct1/metrics.n - metrics.count_top1_percent).abs() < 1e-8).all(),
    'BN-tellers gecontroleerd': all(r['batchnorm_counters_correct'] for r in records),
    'Geen optimizerupdates bij test': all(r['test_optimizer_steps'] == 0 for r in records),
    'Alle gradients voor optimizerstep gecontroleerd': all(r['gradient_checks']['finite_gradient_checks'] == math.ceil(config['train_examples']/config['batch_size']) for r in records),
    'FP32-Adamstaat gecontroleerd': all(r['adam_checks']['moments_fp32_finite'] for r in records),
}
display(pd.DataFrame([{'controle': k, 'geslaagd': bool(v)} for k,v in checks.items()]))
if not all(checks.values()):
    display(Markdown('**Een rapportcontrole faalt: eerst onderzoeken, geen conclusie over MX trekken.**'))
display(pd.DataFrame(report.get('warnings', [])))

## Leer- en tijdverloop
Training bevat dropout en updates, test gebruikt evaluatiemodus. Vergelijk trainings- en testloss daarom niet alsof de uitvoeringscondities identiek zijn. De tijdsraming omvat controles en checkpointopslag; zij is geen hardwarebenchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for phase, data in metrics.groupby('phase'):
    ax.plot(data.epoch, data.count_top1_percent, marker='o', label=phase)
ax.set(xlabel='Epoch', ylabel='Top-1 accuracy (%)', title=f"{config['weight_format']}/A32 - {'synthetisch' if synthetic else 'SVHN train-only'}")
ax.legend(); ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for phase, data in metrics.groupby('phase'):
    ax.plot(data.epoch, data.objective_loss, marker='o', label=phase)
ax.set(xlabel='Epoch', ylabel='Hard-label cross-entropy', title='Loss - een forward/backward per minibatch')
ax.legend(); ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()

In [ ]:
display(times[['epoch','train_seconds','test_seconds','other_seconds','checkpoint_seconds','total_epoch_seconds']])
estimate = read_json('timing_estimate.json')
display(Markdown(f"**Gemeten epochs samen:** {times.total_epoch_seconds.sum()/60:.2f} minuten."))
if estimate.get('available') and not synthetic:
    display(Markdown(f"**Extrapolatie naar 100 epochs:** {estimate['estimated_100_epoch_seconds']/60:.1f} minuten. Nog niet daadwerkelijk 100 epochs gemeten."))
else:
    display(Markdown('Geen bruikbare productie-tijdsraming (nog geen warme epoch of synthetische test).'))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(times.epoch, times.total_epoch_seconds, marker='o')
ax.set(xlabel='Epoch', ylabel='Seconden', title='Totale gemeten epochtijd')
ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()

## Worden de gewichten werkelijk bijgewerkt?
Masterwijzigingen worden per epoch gemeten. Voor de MX-run worden ook veranderingen in element- en schaalcodes geteld. Die aantallen zijn diagnostiek, geen directe maat voor nauwkeurigheid of bandbreedte. Bij `control` zijn de MX-codekolommen bewust leeg.

In [ ]:
display(updates)
display(pd.DataFrame([r['gradient_checks'] | {'epoch': r['epoch']} for r in records]))
display(pd.DataFrame(report.get('mx_exports', {})).T)
# Exportcontroles vergelijken de gedecodeerde tensors. De volledige testset is
# NIET opnieuw vanuit de export gedraaid; zie full_test_forward_rerun=False.

## Optionele gematchte controle
Vul bovenaan `CONTROL_DIR` in en voer opnieuw uit om twee **afzonderlijke eenmodusproeven** te vergelijken. Het oude vijf-precisiecheckpoint is geen gematchte controle: distillatie en gedeelde training verschillen. Deze vergelijking controleert de opgeslagen initialisatie- en samplevolgordehashes. Zij bewijst geen statistische robuustheid of GPU-bitexactheid.

In [ ]:
if CONTROL_DIR is None:
    display(Markdown('Nog geen controlepad opgegeven; dit notebook toont alleen de huidige run.'))
else:
    cr = read_json('run_report.json', Path(CONTROL_DIR))
    ch = read_json('history.json', Path(CONTROL_DIR))
    if config['mode'] != 'mx' or cr['config']['mode'] != 'control':
        raise ValueError('Gebruik dit vergelijkingsvak vanuit de MX-run, met een control-run als tweede pad.')
    shared_keys = ['model','expand','batch_size','train_examples','test_examples','train_split',
                   'seed','lr','weight_decay','scheduler_milestones','loss','activation_bits',
                   'passes_per_training_batch','master_dtype','batchnorm_policy','weight_preprocess']
    mismatch = [k for k in shared_keys if config[k] != cr['config'][k]]
    if mismatch:
        raise ValueError(f'Geen gematcht protocol: {mismatch}')
    initial_equal = report['initial_model_sha256'] == cr['initial_model_sha256']
    same_length = len(records) == len(ch['epochs'])
    order_equal = same_length and all(x['coverage']['train']['order_sha256'] == y['coverage']['train']['order_sha256'] for x,y in zip(records,ch['epochs']))
    same_environment = report['environment'] == cr['environment'] and report['execution_settings'] == cr['execution_settings']
    same_data = report['dataset_files']['files'] == cr['dataset_files']['files']
    same_code = report['package_sha256'] == cr['package_sha256']
    display(pd.DataFrame([{'initialisatie_gelijk':initial_equal,'samplevolgorde_gelijk':order_equal,
                          'omgeving_gelijk':same_environment,'data_gelijk':same_data,'code_gelijk':same_code}]))
    if not all([initial_equal, order_equal, same_environment, same_data, same_code]):
        raise ValueError('Vergelijking niet volledig gematcht; onderzoek de verschillen voordat je conclusies trekt.')
    cm = pd.read_csv(Path(CONTROL_DIR)/'epoch_metrics.csv')
    paired = metrics.query("phase == 'test'")[['epoch','count_top1_percent','objective_loss']].merge(
        cm.query("phase == 'test'")[['epoch','count_top1_percent','objective_loss']], on='epoch', suffixes=('_mx','_control'))
    paired['verschil_procentpunt'] = paired.count_top1_percent_mx - paired.count_top1_percent_control
    display(paired)
    display(Markdown('Vergelijk eerst dezelfde epochs. Het beste checkpoint mag per run anders zijn; kies niet achteraf de gunstigste vergelijking.'))